# ABP 금융 데이터 컬럼 확인

pandas로 CSV를 읽은 뒤, 컬럼명과 컬럼 안의 항목(값)을 확인하는 기초 분석 노트북입니다.

In [1]:
import pandas as pd
from IPython.display import display
from pathlib import Path

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

In [2]:
# CSV 위치: dataset 폴더를 우선 확인하고, 기존 위치도 예비로 확인합니다.
csv_candidates = [Path('dataset/ABP_CONTEST_DATA.csv'), Path('ABP_CONTEST_DATA.csv')]
csv_path = next((path for path in csv_candidates if path.exists()), None)
if csv_path is None:
    raise FileNotFoundError('ABP_CONTEST_DATA.csv 파일을 찾을 수 없습니다.')
df = pd.read_csv(csv_path)

print(f'파일: {csv_path}')
print(f'행 수: {len(df):,}')
print(f'열 수: {df.shape[1]:,}')

파일: ABP_CONTEST_DATA.csv
행 수: 242,574
열 수: 9


## 0. 컬럼명 먼저 확인

## 데이터 컬럼 한글 정리

`dataset/dataset.md`의 코드북을 기준으로 컬럼과 항목을 정리합니다. 단, 코드북의 컬럼명 표기(`TPBUZ_NO`, `AMT`, `CNT`)와 실제 CSV의 표기(`TP_BUZ_NO`, `amt`, `cnt`)에는 차이가 있으므로 실제 CSV 컬럼명을 기준으로 분석합니다.

In [3]:
column_dictionary = pd.DataFrame({
    '컬럼명': [
        'STRD_YYMM', 'SIDO_NM', 'CCG_NM', 'GENDER_CD', 'AGE_CD',
        'TP_BUZ_NO', 'TP_BUZ_NM', 'amt', 'cnt'
    ],
    '한글 의미': [
        '기준년월', '시도명', '시군구명', '성별 코드', '연령대 코드',
        '업종 번호', '업종명', '금액', '건수'
    ],
    '현재 데이터에서 확인되는 구성': [
        '2026년 01월~06월', '17개 시도', '시도별 시군구',
        '1, 2, 3, x', '1, 2, 3, 4, 5, 6, x',
        '11개 업종 번호', '11개 업종', '양의 정수 금액', '양의 정수 건수'
    ],
    '비고': [
        'YYYYMM 형식', '광역자치단체', '기초자치단체',
        '3은 외국인, x는 법인', '1~6은 연령대, x는 법인 및 외국인',
        'TP_BUZ_NM과 매핑 확인 필요', '업종 번호의 한글명',
        '단위: 원', '단위: 건'
    ]
})
display(column_dictionary)

,컬럼명,한글 의미,현재 데이터에서 확인되는 구성,비고
0,STRD_YYMM,기준년월,2026년 01월~06월,YYYYMM 형식
1,SIDO_NM,시도명,17개 시도,광역자치단체
2,CCG_NM,시군구명,시도별 시군구,기초자치단체
3,GENDER_CD,성별 코드,"1, 2, 3, x",x는 미상/기타 가능성; 코드북 필요
4,AGE_CD,연령대 코드,"1, 2, 3, 4, 5, 6, x",x는 미상/기타 가능성; 코드북 필요
5,TP_BUZ_NO,업종 번호,11개 업종 번호,TP_BUZ_NM과 매핑 확인 필요
6,TP_BUZ_NM,업종명,11개 업종,업종 번호의 한글명
7,amt,금액,양의 정수 금액,단위는 원으로 추정되나 확인 필요
8,cnt,건수,양의 정수 건수,거래 또는 이용 건수로 추정


In [4]:
# 데이터에 어떤 컬럼이 있는지 확인합니다.
print(df.columns.tolist())

# 컬럼별 순서와 이름을 표로 확인합니다.
columns_df = pd.DataFrame({
    'column_no': range(1, len(df.columns) + 1),
    'column_name': df.columns
})
display(columns_df)

['STRD_YYMM', 'SIDO_NM', 'CCG_NM', 'GENDER_CD', 'AGE_CD', 'TP_BUZ_NO', 'TP_BUZ_NM', 'amt', 'cnt']


,column_no,column_name
0,1,STRD_YYMM
1,2,SIDO_NM
2,3,CCG_NM
3,4,GENDER_CD
4,5,AGE_CD
5,6,TP_BUZ_NO
6,7,TP_BUZ_NM
7,8,amt
8,9,cnt


## 0-1. 각 컬럼에 어떤 항목이 있는지 확인

In [5]:
# 고유값이 적은 컬럼은 모든 항목을, 많은 컬럼은 상위 20개 항목을 보여줍니다.
for col in df.columns:
    unique_count = df[col].nunique(dropna=False)
    print(f'[{col}] 고유 항목 수: {unique_count:,}')
    display(df[col].value_counts(dropna=False).head(20).rename('count').to_frame())
    print('-' * 60)

[STRD_YYMM] 고유 항목 수: 6


,count
STRD_YYMM,
202605,40696
202606,40511
202604,40507
202603,40433
202601,40267
202602,40160


------------------------------------------------------------
[SIDO_NM] 고유 항목 수: 17


,count
SIDO_NM,
경기도,47005
서울특별시,25792
경상남도,20475
경상북도,20113
전라남도,18357
부산광역시,17046
강원특별자치도,16034
충청남도,14862
전북특별자치도,13056


------------------------------------------------------------
[CCG_NM] 고유 항목 수: 233


...

------------------------------------------------------------
[amt] 고유 항목 수: 36,233


,count
amt,
640000,309
370000,308
430000,301
330000,297
650000,295
420000,292
480000,291
620000,289
270000,288


------------------------------------------------------------
[cnt] 고유 항목 수: 26,906


,count
cnt,
11,1073
12,989
13,956
14,927
16,905
15,881
17,881
18,813
19,809


------------------------------------------------------------


## 주요 항목의 실제 구성

In [6]:
# 기준년월 구성
date_summary = (
    df['STRD_YYMM'].value_counts().sort_index()
      .rename_axis('기준년월').rename('행 수').to_frame()
)
display(date_summary)

# 업종 번호와 업종명의 실제 매핑
business_summary = (
    df[['TP_BUZ_NO', 'TP_BUZ_NM']].drop_duplicates()
      .sort_values('TP_BUZ_NO').reset_index(drop=True)
)
display(business_summary)

# 시도별 시군구 개수
region_summary = (
    df[['SIDO_NM', 'CCG_NM']].drop_duplicates()
      .groupby('SIDO_NM').size().sort_values(ascending=False)
      .rename('시군구 수').to_frame()
)
display(region_summary)

,행 수
기준년월,
202601,40267
202602,40160
202603,40433
202604,40507
202605,40696
202606,40511


,TP_BUZ_NO,TP_BUZ_NM
0,4004,대형할인점
1,4010,편 의 점
2,4020,슈퍼 마켓
3,8001,일반한식
4,8002,갈비전문점
5,8003,한정식
6,8004,일식회집
7,8005,중국음식
8,8006,서양음식
9,8021,스넥


,시군구 수
SIDO_NM,
경기도,47
서울특별시,25
경상북도,23
경상남도,22
전라남도,22
강원특별자치도,18
부산광역시,16
충청남도,16
전북특별자치도,15


## 1. 원본 데이터 샘플

In [7]:
display(df.head(10))
display(df.tail(5))

,STRD_YYMM,SIDO_NM,CCG_NM,GENDER_CD,AGE_CD,TP_BUZ_NO,TP_BUZ_NM,amt,cnt
0,202605,서울특별시,용산구,2,3,8006,서양음식,734430000,38874
1,202605,부산광역시,사상구,2,4,8001,일반한식,424220000,10929
2,202601,경상북도,포항시 북구,1,5,4010,편 의 점,283030000,29848
3,202601,전라남도,여수시,1,6,4010,편 의 점,204060000,20727
4,202606,서울특별시,송파구,1,3,4010,편 의 점,575120000,86780
5,202604,전북특별자치도,군산시,1,5,8021,스넥,106730000,6283
6,202605,경기도,남양주시,1,4,4004,대형할인점,332770000,8755
7,202606,경기도,의왕시,1,4,4004,대형할인점,100640000,3592
8,202604,경상북도,경산시,1,3,4020,슈퍼 마켓,178230000,12081
9,202605,전북특별자치도,익산시,3,5,8001,일반한식,55830000,1385


,STRD_YYMM,SIDO_NM,CCG_NM,GENDER_CD,AGE_CD,TP_BUZ_NO,TP_BUZ_NM,amt,cnt
242569,202603,경상북도,봉화군,x,x,8005,중국음식,310000,15
242570,202603,전라남도,고흥군,x,x,4004,대형할인점,570000,16
242571,202603,경상남도,거창군,x,x,8021,스넥,670000,45
242572,202603,강원특별자치도,영월군,x,x,8021,스넥,240000,15
242573,202603,전라남도,완도군,x,x,8021,스넥,220000,15


## 2. 컬럼명, 자료형, 메모리 사용량

In [8]:
column_info = pd.DataFrame({
    'column': df.columns,
    'dtype': df.dtypes.astype(str).values,
    'non_null': df.notna().sum().values,
    'missing': df.isna().sum().values,
    'missing_rate': (df.isna().mean() * 100).round(2).values,
    'unique': df.nunique(dropna=False).values,
})
display(column_info)
print(f'전체 메모리 사용량: {df.memory_usage(deep=True).sum() / 1024**2:,.2f} MB')

,column,dtype,non_null,missing,missing_rate,unique
0,STRD_YYMM,int64,242574,0,0.00,6
1,SIDO_NM,object,242574,0,0.00,17
2,CCG_NM,object,242574,0,0.00,233
3,GENDER_CD,object,242574,0,0.00,4
4,AGE_CD,object,242574,0,0.00,7
5,TP_BUZ_NO,int64,242574,0,0.00,11
6,TP_BUZ_NM,object,242574,0,0.00,11
7,amt,int64,242574,0,0.00,36233
8,cnt,int64,242574,0,0.00,26906


전체 메모리 사용량: 105.80 MB


## 3. 결측치와 중복 확인

In [9]:
print('컬럼별 결측치')
display(df.isna().sum().sort_values(ascending=False).to_frame('missing_count'))
print(f'완전 중복 행 수: {df.duplicated().sum():,}')
print(f'중복률: {df.duplicated().mean() * 100:.2f}%')

컬럼별 결측치


,missing_count
STRD_YYMM,0
SIDO_NM,0
CCG_NM,0
GENDER_CD,0
AGE_CD,0
TP_BUZ_NO,0
TP_BUZ_NM,0
amt,0
cnt,0


완전 중복 행 수: 0
중복률: 0.00%


## 4. 컬럼별 고유값 확인

범주형 컬럼의 코드와 실제 값을 확인합니다. 고유값이 너무 많은 컬럼은 상위 값만 표시합니다.

In [10]:
for col in df.columns:
    n_unique = df[col].nunique(dropna=False)
    print(f'[{col}] 고유값 {n_unique:,}개')
    if n_unique <= 30:
        display(df[col].value_counts(dropna=False).rename('count').to_frame())
    else:
        display(df[col].value_counts(dropna=False).head(10).rename('count').to_frame())
    print()

[STRD_YYMM] 고유값 6개


,count
STRD_YYMM,
202605,40696
202606,40511
202604,40507
202603,40433
202601,40267
202602,40160



[SIDO_NM] 고유값 17개


,count
SIDO_NM,
경기도,47005
서울특별시,25792
경상남도,20475
경상북도,20113
전라남도,18357
부산광역시,17046
강원특별자치도,16034
충청남도,14862
전북특별자치도,13056



[CCG_NM] 고유값 233개


...

,count
TP_BUZ_NM,
편 의 점,28967
서양음식,28627
슈퍼 마켓,28456
일반한식,28406
스넥,27438
제 과 점,27224
중국음식,26805
일식회집,21714
대형할인점,21529



[amt] 고유값 36,233개


,count
amt,
640000,309
370000,308
430000,301
330000,297
650000,295
420000,292
480000,291
620000,289
270000,288



[cnt] 고유값 26,906개


,count
cnt,
11,1073
12,989
13,956
14,927
16,905
15,881
17,881
18,813
19,809


## 5. 숫자형 컬럼 요약 통계

In [11]:
display(df.describe(include='all').T)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
STRD_YYMM,"242,574.00",NaN,NaN,NaN,"202,603.51",1.71,"202,601.00","202,602.00","202,604.00","202,605.00","202,606.00"
SIDO_NM,242574,17,경기도,47005,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CCG_NM,242574,233,중구,6203,NaN,NaN,NaN,NaN,NaN,NaN,NaN
GENDER_CD,242574,4,1,80554,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AGE_CD,242574,7,6,40326,NaN,NaN,NaN,NaN,NaN,NaN,NaN
TP_BUZ_NO,"242,574.00",NaN,NaN,NaN,"6,739.91","1,897.09","4,004.00","4,020.00","8,004.00","8,006.00","8,301.00"
TP_BUZ_NM,242574,11,편 의 점,28967,NaN,NaN,NaN,NaN,NaN,NaN,NaN
amt,"242,574.00",NaN,NaN,NaN,"71,008,847.07","165,374,742.64","20,000.00","3,440,000.00","16,000,000.00","60,450,000.00","4,247,220,000.00"
cnt,"242,574.00",NaN,NaN,NaN,"3,912.64","8,560.84",11.00,178.00,810.00,"3,330.00","174,529.00"


## 6. 주요 코드 매핑과 데이터 검증

`GENDER_CD`, `AGE_CD`, `TP_BUZ_NO`의 분포와 업종번호-업종명 매핑이 일관적인지 확인합니다.

In [12]:
for col in ['GENDER_CD', 'AGE_CD', 'TP_BUZ_NO', 'TP_BUZ_NM']:
    if col in df.columns:
        print(f'--- {col} ---')
        display(df[col].value_counts(dropna=False).sort_index().to_frame('count'))

if {'TP_BUZ_NO', 'TP_BUZ_NM'}.issubset(df.columns):
    mapping_check = df.groupby('TP_BUZ_NO')['TP_BUZ_NM'].nunique()
    print('업종번호별 업종명 개수')
    display(mapping_check.to_frame('name_count'))
    inconsistent = mapping_check[mapping_check > 1]
    print('매핑 불일치:', '없음' if inconsistent.empty else inconsistent.index.tolist())

--- GENDER_CD ---


,count
GENDER_CD,
1,80554
2,78681
3,70178
x,13161


--- AGE_CD ---


,count
AGE_CD,
1,31293
2,38433
3,39335
4,39762
5,40264
6,40326
x,13161


--- TP_BUZ_NO ---


...

--- TP_BUZ_NM ---


,count
TP_BUZ_NM,
갈비전문점,2860
대형할인점,21529
서양음식,28627
슈퍼 마켓,28456
스넥,27438
일반한식,28406
일식회집,21714
제 과 점,27224
중국음식,26805


업종번호별 업종명 개수


,name_count
TP_BUZ_NO,
4004,1
4010,1
4020,1
8001,1
8002,1
8003,1
8004,1
8005,1
8006,1


매핑 불일치: 없음


## 7. 날짜 및 금액·건수 기본 검증

In [13]:
if 'STRD_YYMM' in df.columns:
    date_values = pd.to_datetime(df['STRD_YYMM'].astype(str), format='%Y%m', errors='coerce')
    print('기간:', date_values.min(), '~', date_values.max())
    print('날짜 변환 실패 건수:', date_values.isna().sum())

for col in ['amt', 'cnt']:
    if col in df.columns:
        print(f'{col} <= 0 건수:', (df[col] <= 0).sum())
        display(df[col].describe().to_frame().T)

기간: 2026-01-01 00:00:00 ~ 2026-06-01 00:00:00
날짜 변환 실패 건수: 0
amt <= 0 건수: 0


,count,mean,std,min,25%,50%,75%,max
amt,"242,574.00","71,008,847.07","165,374,742.64","20,000.00","3,440,000.00","16,000,000.00","60,450,000.00","4,247,220,000.00"


cnt <= 0 건수: 0


,count,mean,std,min,25%,50%,75%,max
cnt,"242,574.00","3,912.64","8,560.84",11.00,178.00,810.00,"3,330.00","174,529.00"


In [14]:
# amt/cnt가 매출액과 이용건수라면 행 단위 평균 결제금액을 확인할 수 있습니다.
if {'amt', 'cnt'}.issubset(df.columns):
    valid_cnt = df['cnt'] > 0
    df['avg_amt_per_cnt'] = pd.NA
    df.loc[valid_cnt, 'avg_amt_per_cnt'] = df.loc[valid_cnt, 'amt'] / df.loc[valid_cnt, 'cnt']
    display(df['avg_amt_per_cnt'].describe(percentiles=[.01, .25, .5, .75, .95, .99]).to_frame().T)

,count,unique,top,freq
avg_amt_per_cnt,"242,574.00","184,167.00","10,000.00",497.00
